# Encoder GUI Smoke

Phase 7F/7G rotary encoder input IP (`axi_encoder_input` / `enc_in_0`) の実機確認 Notebook です。

前提:
- `AudioLabOverlay()` はこの Notebook 内で 1 回だけ attach します。再実行時は既存の `ovl` を再利用します。
- `base.bit` はロードしません。`AudioLabOverlay()` 後に別 Overlay もロードしません。
- HDMI baseline は VESA SVGA `800x600 @ 40 MHz`、compact-v2 GUI は framebuffer 左上 `800x480` です。
- Encoder module の `+` は 3.3V、`GND` は PYNQ 共通 GND。5V は禁止です。
- PMOD JA/JB と外付け PCM1808/PCM5102 path は触りません。

配線:

| Encoder | CLK | DT | SW |
| --- | --- | --- | --- |
| ENC0 | F19 | V10 | V8 |
| ENC1 | W10 | B20 | W8 |
| ENC2 | V6 | Y6 | B19 |

推奨順序:
1. Overlay / IP / codec smoke
2. raw register read
3. live monitor で各 encoder の rotate / short / long を確認
4. 必要なら reverse / swap / debounce を調整
5. synthetic GUI event
6. real encoder -> AppState
7. real encoder -> HDMI GUI loop


In [ ]:
# 1. Imports and AudioLabOverlay attach
# Re-running this cell reuses the existing `ovl` object instead of loading a
# second overlay. To force a reload, restart the kernel first.

import os
import sys
import time
import traceback
from pathlib import Path

PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
for path in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "GUI")):
    if path not in sys.path:
        sys.path.insert(0, path)

from audio_lab_pynq import AudioLabOverlay
from audio_lab_pynq.encoder_input import (
    EncoderInput,
    EncoderEvent,
    EXPECTED_VERSION,
    CONFIG_DEFAULT,
    decode_status,
    unpack_delta,
)
from audio_lab_pynq.encoder_ui import EncoderUiController
from audio_lab_pynq.hdmi_backend import AudioLabHdmiBackend, DEFAULT_WIDTH, DEFAULT_HEIGHT
from audio_lab_pynq.hdmi_effect_state_mirror import HdmiEffectStateMirror
from GUI.compact_v2.state import AppState
from GUI.pynq_multi_fx_gui import render_frame_800x480_compact_v2

if "ovl" in globals():
    print("Reusing existing AudioLabOverlay object. Restart the kernel to reload the bitstream.")
else:
    ovl = AudioLabOverlay()
    print("AudioLabOverlay loaded")

keys = list(getattr(ovl, "ip_dict", {}).keys())
print("ip count:", len(keys))
print("AudioLabOverlay class:", type(ovl).__name__)
print("HdmiEffectStateMirror import OK:", HdmiEffectStateMirror.__name__)


In [ ]:
# 2. Encoder IP / HDMI IP / ADAU1761 smoke
# Expected:
# - encoder_ip_name: enc_in_0/s_axi on the deployed PYNQ 2020.1 image
# - axi_vdma_hdmi / v_tc_hdmi: True
# - VTC GEN_ACTSZ (0x60): 0x02580320
# - ADC HPF: True
# - R19: 0x23

from pynq import MMIO


def is_ip_object(obj):
    return obj is not None and (hasattr(obj, "mmio") or (hasattr(obj, "read") and hasattr(obj, "write")))


def find_encoder_ip_name(overlay):
    ip_dict = getattr(overlay, "ip_dict", {})
    for name in ("enc_in_0/s_axi", "enc_in_0", "axi_encoder_input_0", "axi_encoder_input"):
        if name in ip_dict:
            return name
    for name in ip_dict:
        lower = name.lower()
        if "encoder" in lower or lower.startswith("enc_in") or "enc_" in lower:
            return name
    for name in ("enc_in_0", "axi_encoder_input_0", "axi_encoder_input"):
        if is_ip_object(getattr(overlay, name, None)):
            return name
    raise RuntimeError("encoder IP not found in overlay.ip_dict")

encoder_ip_name = find_encoder_ip_name(ovl)
keys = list(getattr(ovl, "ip_dict", {}).keys())
encoder_keys = [k for k in keys if "encoder" in k.lower() or "enc" in k.lower()]
print("encoder_ip_name:", encoder_ip_name)
print("encoder keys:", encoder_keys)
print("enc_in_0 present:", "enc_in_0" in keys)
print("axi_vdma_hdmi:", "axi_vdma_hdmi" in keys)
print("v_tc_hdmi:", "v_tc_hdmi" in keys)

vtc_desc = ovl.ip_dict["v_tc_hdmi"]
vtc_mmio = MMIO(int(vtc_desc["phys_addr"]), int(vtc_desc["addr_range"]))
vtc_gen_actsz = int(vtc_mmio.read(0x60))
print("VTC GEN_ACTSZ @0x60:", "0x%08X" % vtc_gen_actsz, "expected 0x02580320")

hpf = getattr(ovl, "adc_hpf_enabled", None)
if hpf is None and hasattr(ovl, "codec") and hasattr(ovl.codec, "get_adc_hpf_state"):
    hpf = ovl.codec.get_adc_hpf_state()
print("ADC HPF:", hpf)

try:
    r19 = ovl.read_adc_control()
except Exception:
    r19 = int(getattr(ovl, "codec").R19_ADC_CONTROL[0])
print("R19:", "0x%02X" % int(r19))

assert encoder_ip_name, "encoder IP missing"
assert "axi_vdma_hdmi" in keys, "HDMI VDMA missing"
assert "v_tc_hdmi" in keys, "HDMI VTC missing"
assert vtc_gen_actsz == 0x02580320, "unexpected VTC active size"
assert bool(hpf) is True, "ADC HPF is not enabled"
assert int(r19) == 0x23, "R19_ADC_CONTROL should be 0x23"
print("SMOKE_OK")


In [ ]:
# 3. Encoder raw register read
# Reads VERSION / CONFIG / STATUS / DELTA / COUNT / BUTTON_STATE once.

enc = EncoderInput.from_overlay(ovl, ip_name=encoder_ip_name)
version = enc.read_version()
config = enc.read_config()
status = enc.read_status()
delta_packed = enc.read_delta_packed()
counts = enc.read_counts()
button_state = enc.read_button_state()

print("VERSION       = 0x%08X expected 0x%08X" % (version, EXPECTED_VERSION))
print("CONFIG        = 0x%08X default  0x%08X" % (config, CONFIG_DEFAULT))
print("STATUS        = 0x%08X" % status, decode_status(status))
print("DELTA_PACKED  = 0x%08X" % delta_packed, unpack_delta(delta_packed))
print("COUNT0..2     = %d / %d / %d" % counts)
print("BUTTON_STATE  = 0b%s" % bin(button_state)[2:].zfill(3))

enc.clear_events()
print("After clear_events: STATUS=0x%08X DELTA=%s" % (
    enc.read_status(), unpack_delta(enc.read_delta_packed())))
assert version == EXPECTED_VERSION, "encoder VERSION mismatch; bit/hwh may be stale"
print("RAW_READ_OK")


In [ ]:
# 4. Encoder live monitor
# Turn ENC0/1/2 and press each SW while this runs.
# It uses clear_on_read=False temporarily so it can print raw STATUS / DELTA,
# then clears latches explicitly with CLEAR_EVENTS.

MONITOR_SECONDS = 60
POLL_HZ = 50
PRINT_IDLE_EVERY_S = 1.0

old_config = enc.read_config()
enc.configure(clear_on_read=False)
print("Monitoring for %.1f s at %.1f Hz. Interrupt the cell to stop." % (MONITOR_SECONDS, POLL_HZ))
print("CONFIG for monitor: 0x%08X" % enc.read_config())

t0 = time.time()
last_idle_print = 0.0
total_events = 0
try:
    while time.time() - t0 < MONITOR_SECONDS:
        now = time.time() - t0
        status = enc.read_status()
        packed = enc.read_delta_packed()
        counts = enc.read_counts()
        buttons = enc.read_button_state()
        decoded = decode_status(status)
        deltas = unpack_delta(packed)
        has_activity = any(deltas) or any(decoded["rotate_event"]) or any(decoded["short_press"]) or any(decoded["long_press"])
        if has_activity or (now - last_idle_print) >= PRINT_IDLE_EVERY_S:
            print("t=%6.2f status=0x%08X delta=%s counts=%s buttons=0b%s" % (
                now, status, deltas, counts, bin(buttons)[2:].zfill(3)))
            for i in range(3):
                if deltas[i] or decoded["rotate_event"][i]:
                    print("  enc%d rotate raw_delta=%+d" % (i, deltas[i]))
                    total_events += 1
                if decoded["short_press"][i]:
                    print("  enc%d short_press" % i)
                    total_events += 1
                if decoded["long_press"][i]:
                    print("  enc%d long_press" % i)
                    total_events += 1
            last_idle_print = now
        if has_activity:
            enc.clear_events()
        time.sleep(1.0 / max(1.0, POLL_HZ))
except KeyboardInterrupt:
    print("Interrupted by user")
finally:
    enc.write_config(old_config)
    print("Restored CONFIG: 0x%08X" % enc.read_config())
    print("LIVE_MONITOR_DONE total printed activity events:", total_events)


In [ ]:
# 5. Reverse / swap / debounce config
# If a direction is reversed, set REVERSE_ENC* = True.
# If CLK/DT appear swapped, set SWAP_ENC* = True.
# Keep + on 3.3V and GND common; do not move pins to PMOD JA/JB.

DEBOUNCE_MS = 5
CLEAR_ON_READ = True
SW_ACTIVE_LOW = True
REVERSE_ENC0 = False
REVERSE_ENC1 = False
REVERSE_ENC2 = False
SWAP_ENC0 = False
SWAP_ENC1 = False
SWAP_ENC2 = False


def apply_encoder_config_from_constants():
    cfg = enc.configure(
        debounce_ms=DEBOUNCE_MS,
        clear_on_read=CLEAR_ON_READ,
        sw_active_low=SW_ACTIVE_LOW,
        reverse_direction=(REVERSE_ENC0, REVERSE_ENC1, REVERSE_ENC2),
        clk_dt_swap=(SWAP_ENC0, SWAP_ENC1, SWAP_ENC2),
    )
    print("CONFIG written: 0x%08X" % cfg)
    print("debounce_ms=%d clear_on_read=%s sw_active_low=%s" % (
        DEBOUNCE_MS, CLEAR_ON_READ, SW_ACTIVE_LOW))
    print("reverse:", (REVERSE_ENC0, REVERSE_ENC1, REVERSE_ENC2))
    print("swap:   ", (SWAP_ENC0, SWAP_ENC1, SWAP_ENC2))
    return cfg

try:
    import ipywidgets as widgets
    from IPython.display import display
    debounce_w = widgets.IntSlider(value=DEBOUNCE_MS, min=1, max=50, step=1, description="debounce")
    clear_w = widgets.Checkbox(value=CLEAR_ON_READ, description="clear_on_read")
    sw_w = widgets.Checkbox(value=SW_ACTIVE_LOW, description="sw_active_low")
    rev0_w = widgets.Checkbox(value=REVERSE_ENC0, description="rev0")
    rev1_w = widgets.Checkbox(value=REVERSE_ENC1, description="rev1")
    rev2_w = widgets.Checkbox(value=REVERSE_ENC2, description="rev2")
    swp0_w = widgets.Checkbox(value=SWAP_ENC0, description="swap0")
    swp1_w = widgets.Checkbox(value=SWAP_ENC1, description="swap1")
    swp2_w = widgets.Checkbox(value=SWAP_ENC2, description="swap2")
    button = widgets.Button(description="Apply encoder CONFIG")
    out = widgets.Output()

    def _on_apply(_):
        global DEBOUNCE_MS, CLEAR_ON_READ, SW_ACTIVE_LOW
        global REVERSE_ENC0, REVERSE_ENC1, REVERSE_ENC2
        global SWAP_ENC0, SWAP_ENC1, SWAP_ENC2
        DEBOUNCE_MS = int(debounce_w.value)
        CLEAR_ON_READ = bool(clear_w.value)
        SW_ACTIVE_LOW = bool(sw_w.value)
        REVERSE_ENC0, REVERSE_ENC1, REVERSE_ENC2 = bool(rev0_w.value), bool(rev1_w.value), bool(rev2_w.value)
        SWAP_ENC0, SWAP_ENC1, SWAP_ENC2 = bool(swp0_w.value), bool(swp1_w.value), bool(swp2_w.value)
        with out:
            out.clear_output()
            apply_encoder_config_from_constants()

    button.on_click(_on_apply)
    display(widgets.VBox([
        widgets.HBox([debounce_w, clear_w, sw_w]),
        widgets.HBox([rev0_w, rev1_w, rev2_w]),
        widgets.HBox([swp0_w, swp1_w, swp2_w]),
        button,
        out,
    ]))
    print("ipywidgets available. Use the controls above, or edit constants and call apply_encoder_config_from_constants().")
except Exception as exc:
    print("ipywidgets unavailable (%r). Edit constants above, then run:" % (exc,))
    print("apply_encoder_config_from_constants()")

# Apply once with the constants as written.
apply_encoder_config_from_constants()


In [ ]:
# 6. Synthetic GUI event test
# This does not require physical encoder movement. It verifies that encoder
# events mutate AppState and that the compact-v2 renderer still returns an
# 800x480 RGB frame.

state = AppState()
controller = EncoderUiController(state, overlay=None)
scripted_events = [
    EncoderEvent("rotate", 0, 1, 4),
    EncoderEvent("rotate", 1, 1, 4),
    EncoderEvent("rotate", 2, 2, 8),
    EncoderEvent("short_press", 2),
    EncoderEvent("short_press", 0),
    EncoderEvent("long_press", 0),
    EncoderEvent("long_press", 0),
]

for ev in scripted_events:
    controller.handle_event(ev)
    print("event", ev, "-> selected_effect", state.selected_effect,
          "selected_knob", state.selected_knob,
          "dirty", getattr(state, "value_dirty", None),
          "apply", getattr(state, "apply_pending", None))

frame = render_frame_800x480_compact_v2(state)
print("frame shape:", frame.shape, "dtype:", frame.dtype)
assert tuple(frame.shape) == (480, 800, 3)
print("SYNTHETIC_GUI_EVENT_OK")


In [ ]:
# 7. Real encoder -> AppState test
# This reads the physical encoders and updates AppState only. It does not write
# HDMI and does not apply DSP changes unless encoder-3 short_press clears the
# pending flag in the controller.

APPSTATE_TEST_SECONDS = 45
POLL_HZ = 50

enc.configure(clear_on_read=True)
state = AppState()
controller = EncoderUiController(state, overlay=None)
print("Real encoder -> AppState for %.1f s. Interrupt to stop." % APPSTATE_TEST_SECONDS)
print("Expected: ENC0 rotate effect, ENC0 short on/off, ENC1 rotate knob, ENC2 rotate value.")

t0 = time.time()
last_print = 0.0
try:
    while time.time() - t0 < APPSTATE_TEST_SECONDS:
        now = time.time() - t0
        events = enc.poll(timestamp=now)
        if events:
            controller.handle_events(events)
            print("t=%6.2f events=%s selected_effect=%d selected_knob=%d dirty=%s apply=%s last=%s" % (
                now,
                [(e.kind, e.encoder_id, e.delta, e.raw_delta) for e in events],
                state.selected_effect,
                state.selected_knob,
                getattr(state, "value_dirty", False),
                getattr(state, "apply_pending", False),
                getattr(state, "last_encoder_event", None),
            ))
            last_print = now
        elif now - last_print >= 2.0:
            print("t=%6.2f idle selected_effect=%d selected_knob=%d counts=%s buttons=0b%s" % (
                now, state.selected_effect, state.selected_knob,
                enc.read_counts(), bin(enc.read_button_state())[2:].zfill(3)))
            last_print = now
        time.sleep(1.0 / max(1.0, POLL_HZ))
except KeyboardInterrupt:
    print("Interrupted by user")
finally:
    print("REAL_ENCODER_APPSTATE_DONE")


In [ ]:
# 8. Real encoder -> HDMI GUI control loop
# This is the Notebook equivalent of scripts/run_encoder_hdmi_gui.py.
# It starts the integrated HDMI backend, polls encoders, renders compact-v2,
# writes the 800x480 frame at framebuffer (0,0), and stops VDMA/VTC in finally.

GUI_SECONDS = 120
GUI_FPS = 5
RUN_ENCODER_GUI = True


def stop_encoder_gui():
    global RUN_ENCODER_GUI
    RUN_ENCODER_GUI = False
    print("stop flag set")

enc.configure(clear_on_read=True)
state = AppState()
controller = EncoderUiController(state, overlay=ovl, apply_on_value_change=False)
backend = AudioLabHdmiBackend(ovl, width=DEFAULT_WIDTH, height=DEFAULT_HEIGHT)
backend.start()
print("HDMI backend started", DEFAULT_WIDTH, DEFAULT_HEIGHT)
print("Run stop_encoder_gui() from a later cell if the kernel is free, or interrupt this cell to stop.")
print("Expected: ENC0 effect/on-off, ENC1 parameter/model, ENC2 value/apply.")

period = 1.0 / max(1.0, GUI_FPS)
t0 = time.time()
last_status = 0.0
frames = 0
try:
    while RUN_ENCODER_GUI and (time.time() - t0) < GUI_SECONDS:
        loop_t = time.time()
        events = enc.poll(timestamp=loop_t - t0)
        if events:
            controller.handle_events(events)
            print("events:", [(e.kind, e.encoder_id, e.delta) for e in events])
        state.t = loop_t - t0
        frame = render_frame_800x480_compact_v2(state)
        backend.write_frame(frame, placement="manual", offset_x=0, offset_y=0)
        frames += 1
        if loop_t - last_status >= 2.0:
            status = backend.status()
            errors = backend.errors()
            print("t=%6.2f frames=%d fx=%d knob=%d dirty=%s apply=%s vdma=%s errors=%s" % (
                loop_t - t0, frames, state.selected_effect, state.selected_knob,
                getattr(state, "value_dirty", False), getattr(state, "apply_pending", False),
                status.get("vdma_dmasr"), errors))
            if errors.get("dmainterr") or errors.get("dmaslverr") or errors.get("dmadecerr"):
                print("VDMA error detected; stopping loop")
                break
            last_status = loop_t
        elapsed = time.time() - loop_t
        if elapsed < period:
            time.sleep(period - elapsed)
except KeyboardInterrupt:
    print("Interrupted by user")
finally:
    try:
        print("Final backend status:", backend.status())
        print("Final backend errors:", backend.errors())
    finally:
        backend.stop()
        print("HDMI backend stopped")
print("REAL_ENCODER_HDMI_GUI_DONE frames=", frames)


## Troubleshooting notes

- `enc_in_0 present: False`: deployed bit/hwh が古い可能性があります。`AudioLabOverlay()` が読む `audio_lab_pynq/bitstreams/audio_lab.bit` / `.hwh` と overlays registry の copy を確認してください。
- `VTC GEN_ACTSZ` が `0x02580320` ではない: Phase 6I SVGA `800x600 @ 40 MHz` ではない bitstream がロードされています。kernel restart だけでは直らない場合は PYNQ-Z2 を power cycle してください。
- 回転方向が逆: config セルで該当 `REVERSE_ENC* = True` を設定して apply してください。
- 回しても count が不安定、または逆方向に細かく揺れる: `SWAP_ENC* = True` を試し、必要なら `DEBOUNCE_MS` を 8〜20 ms に上げます。
- SW が常時 pressed / 逆になる: `SW_ACTIVE_LOW` を切り替えてください。典型 module は active-low です。
- short が複数回出る: `DEBOUNCE_MS` を上げてください。
- long_press が出ない: 500 ms 以上押し続けてください。`LONG_PRESS_MS` は RTL 固定です。
- HDMI が白画面になる: `AudioLabOverlay()` 後に別 Overlay をロードしないでください。Phase 6I の 40 MHz rgb2dvi PLL は再 download に弱いため、必要なら power cycle してください。
- `+` は必ず 3.3V。5V に接続しないでください。PMOD JA/JB は使いません。
